In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
pd.set_option('display.max_columns', None)

In [18]:
df=pd.read_csv('gurgaon_properties_post_feature_selection(1).csv')

In [19]:
df.dtypes

property_type       object
sector              object
price              float64
bedRoom              int64
bathroom             int64
balcony             object
agePossession       object
built_up_area      float64
servant room         int64
store room           int64
furnishing_type     object
facilities          object
floor_category      object
dtype: object

In [20]:
df['servant room']=df['servant room'].replace(0,'No')
df['servant room']=df['servant room'].replace(1,'Yes')
df['store room']=df['store room'].replace(0,'No')

df['store room']=df['store room'].replace(1,'Yes')

In [21]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,facilities,floor_category
0,flat,sector 7,0.45,2,2,1,Relatively New,1000.0,No,No,unfurnished,basic,Mid Floor
1,flat,sector 3,0.50,2,2,1,Old Property,722.0,No,No,semifurnished,basic,Low Floor
2,flat,sohna road,0.40,2,2,3,New Property,661.0,No,No,unfurnished,basic,High Floor
3,flat,sector 61,1.47,2,2,2,Relatively New,1333.0,No,No,unfurnished,standard,Low Floor
4,flat,sector 92,0.70,2,2,3,Under Construction,1217.0,No,No,unfurnished,basic,Mid Floor


In [22]:
df.isnull().sum()

property_type      0
sector             0
price              0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
facilities         0
floor_category     3
dtype: int64

In [23]:
df.loc[df['floor_category'].isnull(), 'floor_category'] = 'Low Floor'

In [24]:
df.isnull().sum()

property_type      0
sector             0
price              0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
facilities         0
floor_category     0
dtype: int64

In [25]:
df['floor_category'].value_counts()

floor_category
Mid Floor     1769
Low Floor      927
High Floor     796
Name: count, dtype: int64

In [9]:
df.dtypes

property_type       object
sector              object
price              float64
bedRoom              int64
bathroom             int64
balcony             object
agePossession       object
built_up_area      float64
servant room        object
store room          object
furnishing_type     object
facilities          object
floor_category      object
dtype: object

In [26]:
df.to_csv('Gurgaon_properties_post_feature_selection_3.csv',index=False)

In [27]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [28]:
df1=pd.read_csv('Gurgaon_properties_post_feature_selection_3.csv')

In [17]:
df1.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,facilities,floor_category
0,flat,sector 7,0.45,2,2,1,Relatively New,1000.0,No,No,unfurnished,basic,Mid Floor
1,flat,sector 3,0.50,2,2,1,Old Property,722.0,No,No,semifurnished,basic,Low Floor
2,flat,sohna road,0.40,2,2,3,New Property,661.0,No,No,unfurnished,basic,High Floor
3,flat,sector 61,1.47,2,2,2,Relatively New,1333.0,No,No,unfurnished,standard,Low Floor
4,flat,sector 92,0.70,2,2,3,Under Construction,1217.0,No,No,unfurnished,basic,Mid Floor


In [29]:
df1['floor_category'].value_counts()

floor_category
Mid Floor     1769
Low Floor      927
High Floor     796
Name: count, dtype: int64

In [30]:
X = df1.drop(columns=['price'])
y = df1['price']


In [31]:
y_transformed = np.log1p(y)

In [32]:
X.isnull().sum().sort_values(ascending=False)

property_type      0
sector             0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
facilities         0
floor_category     0
dtype: int64

## ordinal encoding

In [33]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'facilities', 'floor_category','servant room', 'store room']

In [34]:
preprocessor = ColumnTransformer(
     transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom','built_up_area']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ],
    remainder='passthrough'
)

In [35]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])


In [36]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', error_score='raise')

In [37]:
scores.mean(),scores.std()

(0.733476214712818, 0.02607933867114368)

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [39]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['property_type','sector','bedRoom',...,'furnishing_type','facilities', 'floor_category']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. 

In [40]:
y_pred = pipeline.predict(X_test)

In [41]:
y_pred = np.expm1(y_pred)

In [42]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9874296498258837

In [43]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [44]:
def scorer(model_name,model):
    output=[]
    output.append(model_name)
    pipeline=Pipeline([
        ('preprocessor',preprocessor),
        ('regressor',model)
    ])
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [45]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [46]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

model_output

[['linear_reg', 0.733476214712818, 0.9874296498258837],
 ['svr', 0.7569615650424587, 0.8771170771728548],
 ['ridge', 0.7334826903161767, 0.9873131945302885],
 ['LASSO', 0.05747114396064438, 1.5020199190074162],
 ['decision tree', 0.7819721271949521, 0.6549040325384263],
 ['random forest', 0.8839501913489214, 0.4941197270989387],
 ['extra trees', 0.8721405868181108, 0.5477532025544898],
 ['gradient boosting', 0.8774001785749178, 0.5470100589179999],
 ['adaboost', 0.747755957983065, 0.7783424379944448],
 ['mlp', 0.8101480934878182, 0.714147570920452],
 ['xgboost', 0.8936612328167286, 0.49880709503597453]]

In [47]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [48]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.883950,0.494120
10,xgboost,0.893661,0.498807
7,gradient boosting,0.877400,0.547010
6,extra trees,0.872141,0.547753
4,decision tree,0.781972,0.654904
9,mlp,0.810148,0.714148
8,adaboost,0.747756,0.778342
1,svr,0.756962,0.877117
2,ridge,0.733483,0.987313
0,linear_reg,0.733476,0.987430


## OneHotEncoding

In [49]:
columns_to_encode = ['property_type','servant room', 'store room', 'furnishing_type', 'facilities', 'floor_category','balcony']

In [50]:
preprocessor = ColumnTransformer(
     transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1', OneHotEncoder(handle_unknown='ignore',drop='first'), ['sector','agePossession'])
    ],
    remainder='passthrough'
)

In [51]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [52]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', error_score='raise')

C:\Users\manoj\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [53]:
scores.mean()

0.8548997021424295

In [54]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [55]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['property_type','sector','bedRoom',...,'furnishing_type','facilities', 'floor_category']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed thro

In [56]:
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)

In [57]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.7223167268177304

In [58]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [59]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [63]:
model_output = []

for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

C:\Users\manoj\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\manoj\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\manoj\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\manoj\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\manoj\anaconda3\Lib\sit

In [64]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.890962,0.436261
10,xgboost,0.896681,0.475574
1,svr,0.887787,0.497368
5,random forest,0.876893,0.504058
9,mlp,0.881825,0.520381
4,decision tree,0.796574,0.572226
7,gradient boosting,0.860948,0.579840
0,linear_reg,0.854900,0.722317
2,ridge,0.855088,0.727445
8,adaboost,0.718374,0.849396


## Target Encoding

In [1]:
%pip install --upgrade category_encoders

Note: you may need to restart the kernel to use updated packages.


In [65]:
import category_encoders as ce

columns_to_encode = ['property_type','servant room', 'store room', 'furnishing_type', 'facilities', 'floor_category','balcony']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [66]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [67]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [69]:
scores.mean(),scores.std()

(0.824714164769218, 0.027191138576757902)

In [70]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [71]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [72]:
model_dict = {
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'xgboost':XGBRegressor()
}

In [73]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [74]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [75]:
model_df.sort_values(['mae'])

,name,r2,mae
2,extra trees,0.897709,0.453826
3,xgboost,0.900881,0.465180
1,random forest,0.896883,0.475979
0,decision tree,0.815527,0.594895


## hyperparameter tuning

In [76]:
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV

In [77]:
import category_encoders as ce
columns_to_encode = ['property_type', 'balcony', 'furnishing_type', 'agePossession','facilities', 'floor_category', 'servant room', 'store room']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [78]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

### RandomizedSearchCV

In [79]:
param_grid = {
    'regressor__n_estimators': [100, 200, 300, 500, 700],
    'regressor__max_depth': [None, 5, 10, 20, 30],
    'regressor__min_samples_split': [2, 5, 10, 20],
    'regressor__min_samples_leaf': [1, 2, 4, 8],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['sqrt', 'log2']
}

In [80]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(pipeline,param_distributions=param_grid, n_iter=20,scoring='r2',cv=kfold,verbose=4, n_jobs=-1)

random_search.fit(X, y_transformed)
for param, value in random_search.best_params_.items():
    print(f"{param}: {value}")

Fitting 10 folds for each of 20 candidates, totalling 200 fits
regressor__n_estimators: 500
regressor__min_samples_split: 2
regressor__min_samples_leaf: 1
regressor__max_samples: 1.0
regressor__max_features: sqrt
regressor__max_depth: None


In [81]:
pipe=random_search.best_estimator_

In [82]:
random_search.best_score_

0.8944090194265509

In [83]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

y_pred = np.expm1(y_pred)

print(mean_absolute_error(np.expm1(y_test),y_pred))

0.4890420531898838


### GridsearchCv

In [84]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [85]:
param_grid = {
    'regressor__n_estimators': [400, 500, 600],
    'regressor__max_depth': [8, 10, 12],
    'regressor__max_features': ['sqrt'],
    'regressor__max_samples': [0.8, 1.0]
}

In [86]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)

grid_search.fit(X, y_transformed)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

Fitting 10 folds for each of 18 candidates, totalling 180 fits
regressor__max_depth: 12
regressor__max_features: sqrt
regressor__max_samples: 1.0
regressor__n_estimators: 500


In [87]:
final_pipe=grid_search.best_estimator_
grid_search.best_score_

0.8914983575902792

In [88]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

final_pipe.fit(X_train,y_train)

y_pred = final_pipe.predict(X_test)

y_pred = np.expm1(y_pred)

print(mean_absolute_error(np.expm1(y_test),y_pred))

0.4978841502850417


In [89]:
from xgboost import XGBRegressor

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        objective='reg:squarederror',
        random_state=42
    ))
])

In [90]:
param_grid = {
    'regressor__n_estimators': [100, 300, 500,700],
    'regressor__max_depth': [3, 5, 7, 10],
    'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'regressor__subsample': [0.7, 0.8, 1.0],
    'regressor__colsample_bytree': [0.7, 0.8, 1.0],
    'regressor__gamma': [0, 0.1, 0.3],
    'regressor__reg_lambda': [0.1, 1, 10]
}

In [92]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=10,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(pipeline,param_distributions=param_grid, n_iter=20,scoring='r2',cv=kfold,verbose=4, n_jobs=-1)

random_search.fit(X, y_transformed)

Fitting 10 folds for each of 20 candidates, totalling 200 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'regressor__colsample_bytree': [0.7, 0.8, ...], 'regressor__gamma': [0, 0.1, ...], 'regressor__learning_rate': [0.01, 0.05, ...], 'regressor__max_depth': [3, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",4
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator 

In [93]:
for param, value in random_search.best_params_.items():
    print(f"{param}: {value}")

regressor__subsample: 0.8
regressor__reg_lambda: 0.1
regressor__n_estimators: 500
regressor__max_depth: 5
regressor__learning_rate: 0.05
regressor__gamma: 0
regressor__colsample_bytree: 0.8


In [94]:
random_search.best_score_

0.9052173917814249

In [95]:
final_pipe = random_search.best_estimator_

X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

final_pipe.fit(X_train,y_train)

y_pred = final_pipe.predict(X_test)

y_pred = np.expm1(y_pred)

print(mean_absolute_error(np.expm1(y_test),y_pred))

0.46378608494869117


In [96]:
param_grid = {
    'regressor__n_estimators': [400, 500, 600],
    'regressor__max_depth': [6, 7, 8],
    'regressor__learning_rate': [0.03, 0.05, 0.07],
    'regressor__subsample': [0.7, 0.8, 0.9],
    'regressor__colsample_bytree': [0.6, 0.7, 0.8],
    'regressor__reg_lambda': [5, 10, 15]
}

In [ ]:
# from sklearn.model_selection import GridSearchCV
# kfold = KFold(n_splits=10, shuffle=True, random_state=42)

# # Hyperparameter Tuning using GridSearchCV
# grid_search = GridSearchCV(
#     pipeline,
#     param_grid=param_grid,
#     scoring='r2',
#     cv=kfold,
#     verbose=2,
#     n_jobs=-1  # Use all CPU cores
# )

# # Fit the Model
# grid_search.fit(X, y_transformed)

In [ ]:
# for param, value in random_search.best_params_.items():
#     print(f"{param}: {value}")

In [30]:
grid_search.best_score_

0.905476949375131

In [35]:
from sklearn.metrics import r2_score, mean_absolute_error
final_pipe = grid_search.best_estimator_

X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

final_pipe.fit(X_train,y_train)

y_pred = final_pipe.predict(X_test)

y_pred = np.expm1(y_pred)

print(mean_absolute_error(np.expm1(y_test),y_pred))
r2_score(np.expm1(y_test), y_pred)

0.46566410655797974


0.8618123169126395

In [36]:
from sklearn.model_selection import cross_val_score

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(
    final_pipe,
    X,
    y_transformed,
    cv=kfold,
    scoring='r2'
)

print(scores.mean())
print(scores.std())

0.9058401140478297
0.011275588885230867


In [37]:
print(df1['price'].describe())

count    3492.000000
mean        2.430982
std         2.764239
min         0.070000
25%         0.950000
50%         1.505000
75%         2.650000
max        31.500000
Name: price, dtype: float64


In [39]:
from sklearn.metrics import mean_absolute_percentage_error

y_pred = final_pipe.predict(X_test)

y_pred = np.expm1(y_pred)
y_true = np.expm1(y_test)

mape = mean_absolute_percentage_error(y_true, y_pred)

print("MAPE:", mape)

MAPE: 0.2027156554048714


In [1]:
import pandas as pd
import numpy as np

In [2]:
df1=pd.read_csv('Gurgaon_properties_post_feature_selection_3.csv')

In [3]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,r2_score

In [5]:
df1['floor_category'].value_counts()

floor_category
Mid Floor     1769
Low Floor      927
High Floor     796
Name: count, dtype: int64

In [6]:
X = df1.drop(columns=['price'])
y = df1['price']

In [7]:
y_transformed = np.log1p(y)

In [8]:
import category_encoders as ce
columns_to_encode = ['property_type', 'balcony', 'furnishing_type', 'agePossession','facilities', 'floor_category', 'servant room', 'store room']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [9]:
from xgboost import XGBRegressor

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        objective='reg:squarederror',
        random_state=42
    ))
])

In [10]:
param_grid = {
    'regressor__n_estimators': [400, 500, 600],
    'regressor__max_depth': [6, 7, 8],
    'regressor__learning_rate': [0.03, 0.05, 0.07],
    'regressor__subsample': [0.7, 0.8, 0.9],
    'regressor__colsample_bytree': [0.6, 0.7, 0.8],
    'regressor__reg_lambda': [5, 10, 15]
}

In [12]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

# Hyperparameter Tuning using GridSearchCV
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='r2',
    cv=kfold,
    verbose=2,
    n_jobs=-1  # Use all CPU cores
)

# Fit the Model
grid_search.fit(X, y_transformed)

Fitting 10 folds for each of 729 candidates, totalling 7290 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__colsample_bytree': [0.6, 0.7, ...], 'regressor__learning_rate': [0.03, 0.05, ...], 'regressor__max_depth': [6, 7, ...], 'regressor__n_estimators': [400, 500, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scorin

In [15]:
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

regressor__colsample_bytree: 0.6
regressor__learning_rate: 0.03
regressor__max_depth: 8
regressor__n_estimators: 600
regressor__reg_lambda: 5
regressor__subsample: 0.8


In [16]:
grid_search.best_score_

0.9063420385947097

In [17]:
from sklearn.metrics import r2_score, mean_absolute_error
final_pipe = grid_search.best_estimator_

X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

final_pipe.fit(X_train,y_train)

y_pred = final_pipe.predict(X_test)

y_pred = np.expm1(y_pred)

print(mean_absolute_error(np.expm1(y_test),y_pred))
r2_score(np.expm1(y_test), y_pred)

0.46996093624821716


0.8604254845436617